# 09 -- Integrating with LLMs

## What We'll Cover
1. The chatbot with memory pattern
2. Agent memory workflow
3. RAG with Supermemory
4. OpenAI integration example
5. Vercel AI SDK / framework integrations
6. Multi-turn conversation management


## 9.1 The Core Pattern: Profile -> Chat -> Store

Every LLM integration follows the same three-step pattern:

1. **Profile**: Get user context before the LLM call
2. **Chat**: Inject profile into system prompt, call LLM
3. **Store**: Save the conversation for future context

This ensures every conversation is personalized AND improves over time.


In [ ]:
# 9.2 Complete chatbot with memory (OpenAI example)
from supermemory import Supermemory
# import openai  # Uncomment if you have openai installed

class MemoryChatbot:
    """A chatbot that remembers users across conversations."""

    def __init__(self, user_id: str):
        self.sm = Supermemory()
        self.user_id = user_id
        self.container_tag = f"user_{user_id}"
        self.conversation_history = []

    def _build_system_prompt(self, user_message: str) -> str:
        """Build a system prompt enriched with user profile and memories."""
        try:
            profile = self.sm.profile(
                container_tag=self.container_tag,
                q=user_message
            )
            static = "\n".join(f"- {f}" for f in profile.profile.static)
            dynamic = "\n".join(f"- {f}" for f in profile.profile.dynamic)
            memories = "\n".join(
                f"- {r.get('memory', str(r))}"
                for r in profile.search_results.results[:5]
            )
            return f"""You are a helpful, personalized assistant.

User Profile (long-term):
{static if static else 'No profile yet.'}

Current Context (recent activity):
{dynamic if dynamic else 'No recent context.'}

Relevant Memories:
{memories if memories else 'No relevant memories.'}

Use this context to personalize your responses."""
        except Exception as e:
            print(f"Profile lookup failed: {e}")
            return "You are a helpful assistant."

    def chat(self, user_message: str) -> str:
        """Process a user message with memory context."""
        system_prompt = self._build_system_prompt(user_message)
        
        # Call LLM (pseudo-code)
        # response = openai.chat.completions.create(
        #     model="gpt-4o",
        #     messages=[
        #         {"role": "system", "content": system_prompt},
        #         *self.conversation_history[-20:],
        #         {"role": "user", "content": user_message}
        #     ]
        # )
        # assistant_message = response.choices[0].message.content
        
        assistant_message = f"[LLM response enriched with profile context]"
        
        # Store the exchange
        self._store_conversation(user_message, assistant_message)
        return assistant_message

    def _store_conversation(self, user_msg: str, assistant_msg: str):
        """Persist the conversation turn."""
        conv_text = f"user: {user_msg}\nassistant: {assistant_msg}"
        try:
            self.sm.add(content=conv_text, container_tag=self.container_tag)
        except Exception as e:
            print(f"Failed to store: {e}")
        self.conversation_history.append({"role": "user", "content": user_msg})
        self.conversation_history.append({"role": "assistant", "content": assistant_msg})

# Demo
bot = MemoryChatbot(user_id="sarah")
print("MemoryChatbot initialized for user: sarah")
response = bot.chat("Can you help me design a dashboard?")
print(f"User: Can you help me design a dashboard?")
print(f"Bot: {response}")
print("\nThe system prompt was enriched with Sarah's profile context.")


In [ ]:
# 9.3 AI agent with persistent memory
class AgentMemory:
    """An AI agent that remembers tasks, preferences, and outcomes."""

    def __init__(self, agent_id: str):
        self.sm = Supermemory()
        self.agent_id = agent_id
        self.container_tag = f"agent_{agent_id}"

    def get_context(self, task_description: str) -> str:
        """Get relevant context before executing a task."""
        profile = self.sm.profile(
            container_tag=self.container_tag,
            q=task_description
        )
        facts = profile.profile.static + profile.profile.dynamic
        memories = [r.get("memory", str(r)) for r in profile.search_results.results[:3]]
        return "\n".join(f"- {x}" for x in facts + memories)

    def record_result(self, task: str, result: str, success: bool):
        """Record task outcome for future reference."""
        status = "SUCCESS" if success else "FAILED"
        content = f"TASK: {task}\nRESULT ({status}): {result}"
        self.sm.add(
            content=content,
            container_tag=self.container_tag,
            metadata={"type": "task_result", "success": success}
        )
        print(f"[MEMORY] Recorded task: {task[:60]}... -> {status}")

    def record_preference(self, preference: str):
        """Record a user preference."""
        self.sm.add(
            content=f"PREFERENCE: {preference}",
            container_tag=self.container_tag,
            metadata={"type": "preference"}
        )

# Demo agent
agent = AgentMemory(agent_id="devops_bot")
agent.record_preference("User prefers deployments on Tuesdays at 10am PST")
agent.record_result("Deploy v2.3 to staging", "All tests passed", True)
agent.record_result("Database migration", "Failed due to lock timeout", False)

ctx = agent.get_context("deploy new version")
print("\nContext for new deployment task:")
print(ctx)


In [ ]:
# 9.4 RAG (Retrieval-Augmented Generation) with Supermemory
class SupermemoryRAG:
    """RAG pipeline backed by Supermemory as the knowledge store."""

    def __init__(self, kb_name: str = "default_kb"):
        self.sm = Supermemory()
        self.kb_tag = f"kb_{kb_name}"

    def ingest(self, content: str, metadata: dict = None):
        return self.sm.add(
            content=content,
            container_tag=self.kb_tag,
            metadata=metadata or {}
        )

    def retrieve(self, query: str, top_k: int = 5) -> list:
        results = self.sm.search.documents(
            q=query,
            container_tags=[self.kb_tag],
            limit=top_k,
            document_threshold=0.5
        )
        return [r.get("memory", str(r)) for r in results.results]

    def answer(self, query: str) -> str:
        chunks = self.retrieve(query)
        context = "\n\n".join(chunks)
        # Send context + query to LLM
        return f"[Answer based on {len(chunks)} retrieved chunks]"

# Build a knowledge base
rag = SupermemoryRAG(kb_name="company_wiki")
rag.ingest("Acme Corp was founded in 2020. We build AI developer tools.")
rag.ingest("Our flagship product is CodeAssist, with 100K+ users.")
rag.ingest("We use Rust, TypeScript, and Python. Infrastructure on AWS.")

chunks = rag.retrieve("What does Acme Corp do?")
print("Retrieved chunks for 'What does Acme Corp do?':")
for i, chunk in enumerate(chunks, 1):
    print(f"  {i}. {chunk[:100]}...")


In [ ]:
# 9.5 Framework Integrations
print("Vercel AI SDK (TypeScript):")
print('  import { withSupermemory } from "@supermemory/tools/ai-sdk"')
print('  const model = withSupermemory(openai("gpt-4o"), {')
print('    containerTag: "user_123",')
print('    customId: "conv-1"')
print('  })')
print()
print("This automatically:")
print("  1. Injects user profile + memories into every request")
print("  2. Stores conversations for future context")
print("  3. Manages the entire memory lifecycle")
print()
print("Also available for: LangChain, Mastra, n8n, and more.")


## 9.6 Best Practices for LLM Integration

1. **Profile before every LLM call** -- it's fast (~50ms) and dramatically improves quality
2. **Store after every exchange** -- conversations are the fuel for better profiles
3. **Use customId for conversations** -- enables incremental updates without duplication
4. **Limit memory injection** -- 3-5 most relevant memories, not everything
5. **Handle profile failures gracefully** -- fall back to a generic system prompt
6. **Combine profile + search** -- `profile(container_tag, q=user_message)` is most efficient

**Next:** Notebook 10 -- Production Patterns
